# ST Score Restore — Stage 11 V2d P4 Clef Source Qualification

Bu notebook yalnızca **source görüntülerinde** frozen Oemer clef detectorunu, bağımsız öğretmen tarafından işaretlenmiş **213 clef kutusuna** karşı ölçer. Restore modeli ve restored görüntüler bu P4 çalışmasına girmez.

**Colab runtime gereksinimi:** Python 3.11 veya 3.12. Güncel Python 3.13 runtime bu repository tarafından desteklenmez. Colab'da `Runtime > Change runtime type > Runtime Version = 2026.07` (Python 3.12.13) ve `Hardware accelerator = None` seçin.

Primary matching sözleşmesi önceden sabitlenmiştir: greedy one-to-one, IoU >= 0.50. Sonuç TP/FP/FN/precision/recall/F1 üretir; sonuç tek başına detectoru PASS yapmaz.

Çalışma CPU-only ONNX Runtime kullanır ve her sayfa sonucunu Drive'a anında kaydeder; bağlantı koparsa yeniden çalıştırıldığında doğrulanmış sayfalar cache'den devam eder.

**Preflight notu:** Repository unit testleri GitHub CI'da Python 3.11/3.12 üzerinde çalıştırılır. Colab, Oemer için runtime paketleri değiştirildikten sonra bu testleri tekrar çalıştırmaz; bunun yerine exact pinned commit + taxonomy/binding + IoU sözleşmesini doğrudan doğrular.


In [ ]:
# 1) P4 CPU detector runtime — Restore/GPU yok.
import sys, subprocess

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        f'Unsupported Colab Python {sys.version_info.major}.{sys.version_info.minor}. '
        'P4 requires Python 3.11/3.12. Choose Runtime > Change runtime type > '
        'Runtime Version 2026.07 (Python 3.12.13), Hardware accelerator None, then reconnect.'
    )
print('P4 supported Python:', sys.version)

subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'poppler-utils'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'onnxruntime', 'onnxruntime-gpu',
                'opencv-python', 'opencv-python-headless',
                'opencv-contrib-python', 'opencv-contrib-python-headless'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=2.0,<2.3',
                'onnxruntime==1.20.1',
                'opencv-python-headless==4.13.0.92',
                'scipy', 'scikit-learn', 'matplotlib', 'pillow', 'typing-extensions'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'git+https://github.com/BreezeWhite/oemer@dbe2a933d630d0f74805d717960eb259473f5978'], check=True)

verify = r'''import sys
import numpy as np
import cv2
import onnxruntime as ort
from oemer.ete import generate_pred
print('python', sys.version)
print('numpy', np.__version__)
print('opencv', cv2.__version__)
print('onnxruntime', ort.__version__, ort.get_available_providers())
assert (3, 11) <= sys.version_info[:2] < (3, 13)
assert tuple(int(x) for x in np.__version__.split('.')[:2]) < (2, 3)
assert ort.__version__ == '1.20.1'
assert ort.get_available_providers() == ['CPUExecutionProvider'] or ('CPUExecutionProvider' in ort.get_available_providers() and 'CUDAExecutionProvider' not in ort.get_available_providers())
print('Oemer import PASS')
'''
subprocess.run([sys.executable, '-c', verify], check=True)
print('P4 RUNTIME PREFLIGHT PASS')


In [ ]:
# 2) Drive + pinned repository + execution-contract preflight.
# GitHub CI already runs the unit-test suite on Python 3.11/3.12.
# Colab does NOT repeat those tests after Oemer-specific package changes.
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

import json, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/st-score-restore-engine')
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git'
PINNED_CODE_COMMIT = 'a67263e02c5a9ea12fe7f8916267d88ad9882e7f'

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', PINNED_CODE_COMMIT], check=True)

head = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
if head != PINNED_CODE_COMMIT:
    raise RuntimeError(f'Pinned repository HEAD mismatch: {head}')
print('repo pinned HEAD PASS:', head)

sys.path.insert(0, str(REPO / 'src'))

from st_score_restore.stage11_v2d_clef_teacher_completion import (
    validate_clef_taxonomy_amendment,
    validate_clef_teacher_completion_binding,
)
from st_score_restore.stage11_v2d_clef_source_qualification import (
    EXPECTED_CLEF_BOX_COUNT,
    PRIMARY_IOU_THRESHOLD,
)
from st_score_restore.stage11_v2d_clef_source_qualification_colab import (
    run,
    sha256_file,
)

evidence_root = REPO / 'evidence' / 'stage11' / 'v2d'
amendment = json.loads(
    (evidence_root / 'v2d-clef-teacher-taxonomy-amendment.v1.json').read_text(encoding='utf-8')
)
binding = json.loads(
    (evidence_root / 'v2d-clef-box-teacher-completion-binding.v1.json').read_text(encoding='utf-8')
)

amendment_result = validate_clef_taxonomy_amendment(amendment)
binding_result = validate_clef_teacher_completion_binding(binding)

assert amendment_result['status'] == 'pass'
assert binding_result['status'] == 'pass'
assert binding_result['clefBoxCount'] == EXPECTED_CLEF_BOX_COUNT == 213
assert PRIMARY_IOU_THRESHOLD == 0.50

print('taxonomy amendment PASS:', amendment_result)
print('teacher binding PASS:', binding_result)
print('P4 contract PASS: 213 teacher boxes; primary IoU =', PRIMARY_IOU_THRESHOLD)
print('P4 EXECUTION PREFLIGHT PASS')


In [ ]:
# 3) Long source-only measurement. Keep this in the notebook kernel so Colab Drive auth is valid.
from st_score_restore.stage11_v2d_clef_source_qualification_colab import run, sha256_file

result_path = run()
print('P4 RESULT:', result_path)
print('P4 RESULT SIZE:', result_path.stat().st_size)
print('P4 RESULT SHA-256:', sha256_file(result_path))


In [ ]:
# 4) Compact final report. Bu hücre yalnız tamamlanmış result dosyasını okur.
import json
payload = json.loads(result_path.read_text(encoding='utf-8'))
print('\n===== P4 CLEF SOURCE QUALIFICATION SUMMARY =====')
print('teacher truth:', payload['teacherTruth']['sha256'])
print('IoU:', payload['matching']['primaryIouThreshold'])
print('pooled:', payload['pooledMetrics'])
print('family metrics:')
for family, metrics in payload['sourceFamilyMetrics'].items():
    print(' ', family, metrics)
print('detectorQualified:', payload['decisionBoundary']['detectorQualified'])
print('qualificationDecisionRequired:', payload['decisionBoundary']['qualificationDecisionRequired'])
print('semanticPreservationEstablished:', payload['decisionBoundary']['semanticPreservationEstablished'])
print('SAVED:', result_path)
print('SHA-256:', sha256_file(result_path))


## Beklenen bitiş

Son hücrede `SAVED:` ve `SHA-256:` görülmelidir. Oluşan `v2d_clef_source_qualification_result.json` dosyasını ChatGPT konuşmasına yükleyin. Detector PASS/REVIEW_ONLY/REJECT kararı bu ölçümden **sonra ayrı** verilecektir.
